Lab 3

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats


In [ ]:
# grab the data from online, Excel is fine
url = 'https://www.qogdata.pol.gu.se/data/qog_bas_cs_jan24.xlsx'
df = pd.read_excel(url)

In [ ]:
df.head()

,ccode,cname,ccode_qog,cname_qog,ccodealp,ccodecow,version,ajr_settmort,atop_ally,atop_number,...,wvs_imprel,wvs_pmi12,wvs_psarmy,wvs_psdem,wvs_psexp,wvs_pssl,wvs_relacc,wvs_satfin,wvs_subh,wvs_trust
0,4,Afghanistan,4,Afghanistan,AFG,700.0,QoGBasCSjan24,4.540098,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8,Albania,8,Albania,ALB,339.0,QoGBasCSjan24,NaN,1.0,8.0,...,2.869328,NaN,1.596485,3.849031,3.475513,1.744196,NaN,NaN,3.488758,0.027857
2,12,Algeria,12,Algeria,DZA,615.0,QoGBasCSjan24,4.359270,1.0,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20,Andorra,20,Andorra,AND,232.0,QoGBasCSjan24,NaN,1.0,2.0,...,2.034930,2.710393,1.336049,3.681363,2.635721,1.830491,1.751004,6.561316,4.089642,0.255744
4,24,Angola,24,Angola,AGO,540.0,QoGBasCSjan24,5.634789,1.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**1--  Run a simple bivariate regression, and interpret your results. (Did the results fit your expectations? Why? Why not?)**


Dependent Variable: GDP per capita (2015 US dollars) (wdi_gdpcapcon2015).  You can see the distribution of it here.

In [ ]:
df[['wdi_gdpcapcon2015']].describe()

,wdi_gdpcapcon2015
count,188.000000
mean,13726.184306
std,21112.612680
min,263.360992
25%,1939.369629
50%,5395.598633
75%,15333.187744
max,166910.625000


Independent Variable: Corruption Perceptions Index (ti_cpi), Scale of 0-100 where 0 equals the highest level of perceived corruption and 100 equals the lowest level of perceived corruption.  This is the description of the corruption index.  



In [ ]:
df[['ti_cpi']].describe()

,ti_cpi
count,179.000000
mean,43.262570
std,18.771341
min,12.000000
25%,29.000000
50%,40.000000
75%,56.000000
max,88.000000


I expect that there is a relationship where economies perceived as less corrupt (by their populaces) tend to have higher GDP per capita.  (Notice the subset=df['p_polity2'].notna() element below, which restricts the analysis to countries that are not missing on this other variable, vdem_corr, which I am going to add in the next analysis, and needs to be not missing.)

In [ ]:
corruption_gdp = smf.ols(formula = 'wdi_gdpcapcon2015~ti_cpi', data = df, subset=df['p_polity2'].notna()).fit()
print (corruption_gdp.summary())

                            OLS Regression Results                            
Dep. Variable:      wdi_gdpcapcon2015   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.635
Method:                 Least Squares   F-statistic:                     277.4
Date:                Thu, 16 Oct 2025   Prob (F-statistic):           1.32e-36
Time:                        19:15:49   Log-Likelihood:                -1718.8
No. Observations:                 160   AIC:                             3442.
Df Residuals:                     158   BIC:                             3448.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -2.116e+04   2224.627     -9.513      0.0

Coefficient for Corruption Perceptions Index is 794, which indicates that for each additional point in the Corruption Perceptions Index suggests lower perceived corruption, such that the GDP per capita increases by approximately 794 USD. The model is statistically significant with p < 0.001. The R-squared of 0.64 indicats a strong association.

Indeed, the regression suggests a relationship where economies perceived as less corrupt tend to have higher GDP per capita.

**2--  Add an additional variable that might mediate or partly "explain" the initial association from that simple regression above -- and explain your results. Did it work out? Yes? No?**


But perceived corruption might be more prevalent in less democratic societies, so I include the p_polity2 variable, which measures how −10 (strongly autocratic) to +10 (strongly democratic) countries are.  I would expect, with this new variable added, it will absorb some of the effect of perceived corruption because democratic institutions reduce corruption by making leaders accountable to voters, opposition parties, and the media.

In [ ]:
df[['p_polity2']].describe()

,p_polity2
count,165.000000
mean,4.066667
std,6.174429
min,-10.000000
25%,-1.000000
50%,7.000000
75%,9.000000
max,10.000000


How do my two independent variables correlate?  They correlate meaningfully (+0.42).

In [ ]:
df_filtered = df[['ti_cpi', 'p_polity2']].dropna()
stats.pearsonr(df_filtered['ti_cpi'], df_filtered['p_polity2'])


PearsonRResult(statistic=np.float64(0.42400879129703695), pvalue=np.float64(1.3857922214726694e-08))

In [ ]:
corruption_actual_gdp = smf.ols(formula='wdi_gdpcapcon2015 ~ ti_cpi +  p_polity2', data=df).fit()
print(corruption_actual_gdp.summary())

                            OLS Regression Results                            
Dep. Variable:      wdi_gdpcapcon2015   R-squared:                       0.646
Model:                            OLS   Adj. R-squared:                  0.642
Method:                 Least Squares   F-statistic:                     143.4
Date:                Thu, 16 Oct 2025   Prob (F-statistic):           3.72e-36
Time:                        19:22:29   Log-Likelihood:                -1716.7
No. Observations:                 160   AIC:                             3439.
Df Residuals:                     157   BIC:                             3449.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -2.154e+04   2211.439     -9.741      0.0

Corruption Perceptions Index has a coefficient of 834, indicating that a 1 point increase in better perceived corruption (where it goes up to 100) is associated with an increase of $834 in GDP per capita, net of actual corruption.

The Polity2 score (p_polity2), which measures regime type ranging from -10 (autocracy) to +10 (full democracy), has a negative coefficient of -320.7. This suggests that, controlling for perceived corruption, countries with higher democracy scores (more democratic) are associated with lower GDP per capita on average.  This result is barely statistically significant (with p-value barely under 0.05).  Hmmmmmm?

The adjusted R-squared is 0.642, which is just the slightest bit larger than the earlier model, only with perceived corruption.  This additional variable adds almost nothing in terms of predictive accuracy of the model.

This result seems somewhat counterintuitive and suggests that while perceived corruption and democracy are meaningfully correlated, when they differ, their effects also differ, with lower perceived corruption being highly beneficial for GDP, but higher democratic scores actually lowering GDP, though at barely statistically significant levels.  

The biggest finding here is that it is clearly the case that democratic scores do not mediate or absorb any of the effect of perceived corruption on GDP per capita.  At any level of democracy, perceived corruption is quite deleterious to GDP.

**3-- More on extreme combinations.  Find the top 5 entities that are ranked at the top on one variable and ranked at the bottom on another variable.  Interpret your results.**

I am going to look to countries that are very high in their human capital formation, but do not appear to have very much political participation among their citizenry.

In [ ]:
df['human_capital_rank'] = df['pwt_hci'].rank(ascending=False)
df['political_participation_rank'] = df['bti_pp'].rank(ascending=False)

extreme_comb = df[['cname', 'human_capital_rank', 'political_participation_rank']].copy()
extreme_comb['rank_difference'] = extreme_comb['political_participation_rank'] - extreme_comb['human_capital_rank']
extreme_sorted = extreme_comb.sort_values(by='rank_difference', ascending=False).head(5)

extreme_sorted

,cname,human_capital_rank,political_participation_rank,rank_difference
155,Singapore,1.0,93.0,92.0
141,Qatar,36.0,115.0,79.0
170,Tajikistan,53.0,123.5,70.5
143,Russian Federation (the),26.0,93.0,67.0
86,Kazakhstan,47.0,107.5,60.5


These results highlight a significant difference in these countries between the level of human capital and political participation. This could suggest a potential inefficiency or barrier in translating high human capital into political engagement.  It is interesting that so many of these countries were formerly communist countries.